**TRABAJO PRACTICO - PIPELINE CON AEROLINEA 
ARQUITECTURA BRONZE, SILVER Y GOLD CON PYSPARK Y DELTA LAKE**


En este NOtebook se contruye un pipeline de datos para una aerolinea de datos para uma aerolinea arquitectura por capas:
- **Bronze**: imgesta de archivos duente 
- **Silver**: Limpieza, tipificacion y deduplificacion 
- **Gold**: generacion de KPIs de negocio 

**1. Importamos Librerias**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession



**2.Creación de Catalogo y Volumen**


In [0]:
#Configuracion

CATALOG = "airline_mantenimiento"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
VOLUME_NAME = "landing"

VUELOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/vuelos_diarios.csv"
AERONAVES_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/aeronaves.csv"
AEROPUERTOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/aeropuertos.csv"
MANTENIMIENTO_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/mantenimientos_rds.csv"

spark = SparkSession.builder.getOrCreate()

In [0]:
# Creacion de Catalogo, Esquema y Volumen

#Creamos el catalogo
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

#Creamos los esquemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# Creamos el volumen
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

          

**3. CAPA BRONZE**

basicos y se almacena como tablas DELTA


In [0]:
#Leemos vuelos 
vuelos_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(VUELOS_PATH)     
 )
 #Leemos aeronaves
aeronaves_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(AERONAVES_PATH)     
 )
 #Leemos aeropuertos
aeropuertos_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(AEROPUERTOS_PATH)     
 )
 #Leemos mantenimientos
mantenimientos_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(MANTENIMIENTO_PATH)
)


In [0]:
# Validación

print("Filas vuelos:", vuelos_raw.count())
print("Filas areonaves:", aeronaves_raw.count())
print("Filas aeropuertos:", aeropuertos_raw.count())
print("Filas mantenimiento:", mantenimientos_raw.count())


In [0]:
# Normalizar Vuelos Bronze

##Normalizar vuelos bronze
vuelos_bronze = (
    vuelos_raw
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("vuelo_id", F.col("vuelo_id").cast("string"))
    .withColumn("origen_id", F.col("origen_id").cast("string"))
    .withColumn("destino_id", F.col("destino_id").cast("string"))
    .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("estado", F.col("estado").cast("string"))
    .withColumn("duracion_min", F.col("duracion_min").cast("int"))
    .withColumn("ingestion_time", F.current_timestamp())
)

In [0]:
#Normalizar aeronaves bronce

aeronaves_bronze = (
    aeronaves_raw
    .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("modelo", F.col("modelo").cast("string"))
    .withColumn("fabricante", F.col("fabricante").cast("string"))
    .withColumn("anio_fabricacion", F.col("anio_fabricacion").cast("string"))
    .withColumn("ingestion_time", F.current_timestamp())
)

In [0]:

##Normalizar aeropuertos bronce

aeropuertos_bronze = (
    aeropuertos_raw
    .withColumn("aeropuerto_id", F.col("aeropuerto_id").cast("string"))
    .withColumn("nombre", F.col("nombre").cast("string"))
    .withColumn("ciudad", F.col("ciudad").cast("string"))
    .withColumn("pais", F.col("pais").cast("string"))
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("lon", F.col("lon").cast("double"))
    .withColumn("ingestion_time", F.current_timestamp())
)


In [0]:
#Normalizar mantenimientos bronce

mantenimientos_bronze = (
    mantenimientos_raw
    .withColumn("mantenimiento_id", F.col("mantenimiento_id").cast("string"))
    .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("tipo", F.col("tipo").cast("string"))
    .withColumn("costo_usd", F.col("costo_usd").cast("double"))
    .withColumn("duracion_hr", F.col("duracion_hr").cast("double"))
    .withColumn("ingestion_time", F.current_timestamp())
)

In [0]:
#Guardar tablas Bronze
vuelos_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.vuelos")
aeronaves_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.aeronaves")
aeropuertos_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.aeropuertos")
mantenimientos_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.mantenimientos")

In [0]:
# validacion de tablas
spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.vuelos").display()


 **4. CAPA SILVER**

In [0]:
# Leemos las tablas bronze

vuelos_slv.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.vuelos")

aeropuertos_slv.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos")

aeronaves_slv.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves")

mantenimientos_slv.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos")



In [0]:
#Transformaciones Silver vuelos
vuelos_slv = (
    vuelos_bz
    .select("vuelo_id", 
            "fecha", 
            "origen_id", 
            "destino_id", 
            "aeronave_id", 
            "estado", 
            "duracion_min")
    .withColumn("fecha",F.to_date(F.col("fecha"))) # Convertir a fecha
    .withColumn("duracion_min",F.col("duracion_min").cast("int")) # Convertir a entero
    .dropDuplicates(["vuelo_id"])
)

# Transformaciones silver aeropuertos
aeropuertos_slv = (
    aeropuertos_bz
    .select(
        "aeropuerto_id",
        "nombre",
        "ciudad",
        "pais",
        "lat",
        "lon"
    )
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("lon", F.col("lon").cast("double"))
    .filter(F.col("lat").isNotNull() & F.col("lon").isNotNull())
    .filter(
        (F.col("lat") >= -90) & (F.col("lat") <= 90) &
        (F.col("lon") >= -180) & (F.col("lon") <= 180)
    )
    .dropDuplicates(["aeropuerto_id"])
)

# Transformaciones silver aeronaves
aeronaves_slv = (
    aeronaves_bz
    .select(
        "aeronave_id",
        "modelo",
        "fabricante",
        "anio_fabricacion"
    )
    .withColumn("anio_fabricacion", F.col("anio_fabricacion").cast("int"))
    .filter(F.col("anio_fabricacion").isNotNull())
    .dropDuplicates(["aeronave_id"])
)

# Transformaciones silver mantenimientos
mantenimientos_slv = (
    mantenimientos_bz
    .select(
        "mantenimiento_id",
        "aeronave_id",
        "fecha",
        "tipo",
        "costo_usd",
        "duracion_hr"
    )
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("costo_usd", F.col("costo_usd").cast("double"))
    .withColumn("duracion_hr", F.col("duracion_hr").cast("double"))
    .dropDuplicates(["mantenimiento_id"])
)

In [0]:
# Validación de Calidad
print("Nulos vuelo_id:", vuelos_slv.filter(F.col("vuelo_id").isNull()).count())
print("Duplicados vuelos:", vuelos_slv.groupBy("vuelo_id").count().filter(F.col("count") > 1).count())


In [0]:
# validamos silver
spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.vuelos").display()

**Capa Gold**


- KPI de puntualidad de vuelos
Se genera una tabla analítica con métricas de puntualidadpor modelo, fabricante, país de origen y periodo.

In [0]:
##Leer Silver para puntualidad
df_vuelos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.vuelos")
df_aeronaves = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves")
df_aeropuertos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos")

In [0]:
##Preparar aeropuertos de origen
df_aeropuertos_origen = df_aeropuertos.select(
    F.col("aeropuerto_id").alias("origen_id"),
    F.col("nombre").alias("origen_nombre"),
    F.col("ciudad").alias("origen_ciudad"),
    F.col("pais").alias("origen_pais"),
   )

In [0]:
#Validación
df_aeropuertos_origen.show(5)

In [0]:
# Enriquecer vuelos
df_vuelos_enriquecido = (
    df_vuelos
    .join(df_aeropuertos_origen, on="origen_id", how="left")
    .join(df_aeronaves, on="aeronave_id", how="left")
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
)

#visualizacion
display(df_vuelos_enriquecido)

In [0]:
#calcular KPI de puntualidad
df_kpi_puntualidad = (
    df_vuelos_enriquecido
    .groupBy("modelo", "fabricante", "origen_pais", "mes", "anio")
    .agg(
        F.count("vuelo_id").alias("total_vuelos"),
        F.count(F.when(F.col("estado") == "a_tiempo", True)).alias("vuelos_a_tiempo"),
        F.count(F.when(F.col("estado") == "retrasado", True)).alias("vuelos_retrasados"),
        F.count(F.when(F.col("estado") == "cancelado", True)).alias("vuelos_cancelados"),
        F.round(F.avg("duracion_min"),2).alias("duracaion_promedio_min")
    ))
#Validación
display(df_kpi_puntualidad)

In [0]:
#Guardar Gold Puntualidad
df_kpi_puntualidad .write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_puntualidad_vuelos")

**6. Capa Gold - KPI** de costos de mantenimiento
Se genera una tabla analítica con métricas de mantenimiento por aeronave, modelo, fabricante , tipo y periodo.

In [0]:
df_mantenimiento = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos")

In [0]:
# Enriquecer mantenimiento
df_mantenimientos_enriquecido = (
    df_mantenimiento
    .join(df_aeronaves, on="aeronave_id", how="left")
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
)

#validación
display(df_mantenimientos_enriquecido)

In [0]:
# Calcular KPI de mantenimiento
df_kpi_mantenimiento = (
    df_mantenimientos_enriquecido
    .groupBy("aeronave_id", "modelo", "fabricante", "tipo", "mes", "anio")
    .agg(
        F.count("mantenimiento_id").alias("total_mantenimientos"),
        F.round(F.avg("costo_usd"), 2).alias("costo_promedio_usd"),
        F.round(F.sum("costo_usd"), 2).alias("costo_total_usd"),
        F.round(F.avg("duracion_hr"), 2).alias("duracion_promedio_hr"),
        F.round(F.sum("duracion_hr"), 2).alias("duracion_total_hr")
    )
)

#visualizacion
display(df_kpi_mantenimiento)

In [0]:
# Guardar Gold Mantenimiento
df_kpi_mantenimiento.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_costo_mantenimiento")